## 1. Basic Tasks

**1. Create a catalog cyntexa_dev and a schema sales within it.**

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS cyntexa_dev

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS cyntexa_dev.sales

**2. Create a managed table sales.orders_raw with at least 5 columns and insert 10 sample rows.**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cyntexa_dev.sales.orders_raw (
    order_id STRING,
    customer_id STRING,
    product_id STRING,
    order_date DATE,
    amount DOUBLE,
    status STRING,
    region STRING
);

In [0]:
%sql
INSERT INTO cyntexa_dev.sales.orders_raw (order_id, customer_id, product_id, order_date, amount, status, region) VALUES
('O001', 'C1000A', 'P101', '2026-08-01', 1250.00, 'completed', 'North'),
('O002', 'C2000A', 'P102', '2026-08-02', 850.50, 'pending', 'South'),
('O003', 'C3000A', 'P103', '2026-08-03', 2300.00, 'completed', 'East'),
('O004', 'C4000A', 'P104', '2026-08-04', 450.75, 'cancelled', 'West'),
('O005', 'C5000A', 'P105', '2026-08-05', 1750.25, 'completed', 'North'),
('O006', 'C6000A', 'P101', '2026-08-06', 920.00, 'completed', 'South'),
('O007', 'C7000A', 'P106', '2026-08-07', 3100.50, 'pending', 'East'),
('O008', 'C8000A', 'P102', '2026-08-08', 675.00, 'completed', 'West'),
('O009', 'C9000A', 'P107', '2026-08-09', 1450.75, 'cancelled', 'North'),
('O010', 'C10000A', 'P108', '2026-08-10', 2750.00, 'completed', 'South');

**3. Create a view sales.orders_view that selects only completed orders.**

In [0]:
%sql
CREATE OR REPLACE VIEW cyntexa_dev.sales.orders_view AS
SELECT * FROM cyntexa_dev.sales.orders_raw 
WHERE status = 'completed';

**4. (Data Analyst) Explore samples.bakehouse or samples.tpch and run 3 exploratory SELECT queries.**

In [0]:
%sql
SELECT * FROM samples.bakehouse.sales_customers

In [0]:
%sql
SELECT COUNT(*) AS total_customers, country FROM samples.bakehouse.sales_customers GROUP BY country;

In [0]:
%sql
SELECT sc.customerID, CONCAT(sc.first_name, ' ', sc.last_name) AS customerName, st.quantity, st.unitPrice, st.totalPrice
FROM samples.bakehouse.sales_customers AS sc
JOIN samples.bakehouse.sales_transactions AS st
ON sc.customerID = st.customerID
WHERE sc.country = 'USA'
ORDER BY st.totalPrice DESC

In [0]:
%sql
SELECT * FROM samples.tpch.customer

In [0]:
%sql
SELECT COUNT(*) AS total_customers, c_mktsegment FROM samples.tpch.customer GROUP BY c_mktsegment;

In [0]:
%sql
SELECT c.c_name, c.c_nationkey, c.c_mktsegment, o.o_orderkey, o.o_orderstatus, o.o_totalprice, o.o_orderdate, o.o_comment
FROM samples.tpch.customer AS c
JOIN samples.tpch.orders o ON c.c_custkey = o.o_custkey
WHERE c.c_mktsegment = 'AUTOMOBILE'
ORDER BY o.o_orderdate;

## 2. Intermediate Tasks

**5. Write a SQL UDF that masks the last 4 digits of a customer_id or email column, and apply it in a
SELECT against orders_raw.**

In [0]:
%sql
SELECT 
order_id, 
CONCAT(LEFT(customer_id, LENGTH(customer_id)-4), '****') AS masked_customer_id, 
product_id, 
order_date, 
amount, 
status, 
region
FROM cyntexa_dev.sales.orders_raw;

**6. Create an external table pointing at a cloud storage/DBFS path and use DESCRIBE EXTENDED to
compare its LOCATION and table type against the managed table.**

In [0]:
%sql
-- 2. Create an External Table
-- Replace the path below with your cloud URI (s3://, abfss://, gs://) or DBFS path (dbfs:/tmp/orders_external_data)
CREATE OR REPLACE TABLE cyntexa_dev.sales.orders_external (
    order_id INT,
    product_name STRING,
    amount DOUBLE
)
LOCATION 'dbfs:/tmp/orders_external_data';

CREATE OR REPLACE TABLE hive_metastore.default.orders_external (
    order_id INT,
    product_name STRING,
    amount DOUBLE
)
LOCATION 'dbfs:/tmp/orders_external_data';

* **No Cloud Account Integration:** The Free Community Edition runs entirely on Databricks-hosted infrastructure. We cannot attach our own AWS S3, Azure ADLS, or GCP GCS cloud accounts to configure the required Storage Credentials and External Locations.
* **DBFS Scheme Blocked by Unity Catalog:** Unity Catalog strictly disallows creating external tables using the `dbfs:/` scheme (`UC_FILE_SCHEME_FOR_TABLE_CREATION_NOT_SUPPORTED`). In Unity Catalog, external tables require cloud URIs like `s3://` or `abfss://`.
* **Hive Metastore is Disabled:** Legacy workspace features that previously allowed `dbfs:/` external tables via `hive_metastore` are completely disabled in modern Community Edition workspaces (`UC_HIVE_METASTORE_DISABLED_EXCEPTION`).
* **Volumes Cannot Be Table Locations:** The `/Volumes/...` feature in the Free Edition is designed solely for raw file storage and direct file querying (`read_files()`), not as a target path for the `LOCATION` parameter in `CREATE TABLE`.
* **Missing Admin Privileges for External Locations:** we don't have access to the account console or Metastore Admin privileges required to run `CREATE STORAGE CREDENTIAL` or `CREATE EXTERNAL LOCATION`.

**7. (Data Analyst) Build a second view joining orders_view with a customers table/view and calculate
total spend per customer.**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cyntexa_dev.sales.customers (
    customer_id STRING,
    customer_name STRING,
    email STRING,
    country STRING,
    region STRING,
    created_at TIMESTAMP
);

In [0]:
%sql
INSERT INTO cyntexa_dev.sales.customers (customer_id, customer_name, email, country, region, created_at) VALUES
('C1000A',  'Aarav Sharma',   'aarav.sharma@example.com',   'India', 'North', '2026-01-10 09:30:00'),
('C2000A',  'Deepa Nair',     'deepa.nair@example.com',     'India', 'South', '2026-01-15 11:15:00'),
('C3000A',  'Rohan Mukherjee','rohan.m@example.com',        'India', 'East',  '2026-02-01 14:00:00'),
('C4000A',  'Pooja Patel',    'pooja.patel@example.com',    'India', 'West',  '2026-02-20 16:45:00'),
('C5000A',  'Vikram Verma',   'vikram.v@example.com',       'India', 'North', '2026-03-05 10:20:00'),
('C6000A',  'Ananya Iyer',    'ananya.iyer@example.com',    'India', 'South', '2026-03-18 12:10:00'),
('C7000A',  'Debasish Roy',   'debasish.roy@example.com',   'India', 'East',  '2026-04-12 15:30:00'),
('C8000A',  'Kavita Joshi',   'kavita.j@example.com',       'India', 'West',  '2026-05-01 08:50:00'),
('C9000A',  'Sameer Khan',    'sameer.khan@example.com',    'India', 'North', '2026-05-25 17:05:00'),
('C10000A', 'Siddharth Rao',  'siddharth.rao@example.com',  'India', 'South', '2026-06-14 13:40:00');

In [0]:
%sql
CREATE OR REPLACE VIEW cyntexa_dev.sales.customer_order_view AS
SELECT c.customer_id, c.customer_name, SUM(o.amount) AS total_spent
FROM cyntexa_dev.sales.customers c
INNER JOIN cyntexa_dev.sales.orders_view o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name;

## 3. Advanced Tasks

**8. Design a full three-level namespace plan for Cyntexa (catalogs for dev/staging/prod, schemas per
business domain) and justify the structure in a short writeup.**

## Cyntexa Unity Catalog 3-Level Namespace Architecture

### 1. Hierarchy Overview

Databricks Unity Catalog enforces a 3-level namespace structure: `catalog.schema.table_or_view`.

For Cyntexa I have designed a 3 level namespace, which covers the three environments as Catalogs cyntexa_dev, cyntexa_stage and cyntexa_prod, and business domains as Schemas such as sales, engineering, hr etc.

```text
Unity Catalog Metastore
│
├── cyntexa_dev (Catalog: Development & Sandbox)
│   ├── sales           (Schema)
│   ├── engineering     (Schema)
│   ├── finance         (Schema)
│   ├── hr              (Schema)
│   ├── marketing       (Schema)
│   ├── social media    (Schema)
│   └── IT              (Schema)
│
├── cyntexa_stage (Catalog: Staging & UAT Validation)
│   ├── sales           (Schema)
│   ├── engineering     (Schema)
│   ├── finance         (Schema)
│   ├── hr              (Schema)
│   ├── marketing       (Schema)
│   ├── social media    (Schema)
│   └── IT              (Schema)
│
└── cyntexa_prod (Catalog: Production / Golden Records)
    ├── sales           (Schema)
    ├── engineering     (Schema)
    ├── finance         (Schema)
    ├── hr              (Schema)
    ├── marketing       (Schema)
    ├── social media    (Schema)
    └── IT              (Schema)

```

---

## 2. Architectural Justification

* **Complete Environment Parallelism:** I have created identical schemas across `dev`, `stage`, and `prod` that allows data pipelines to move through CI/CD via simple parameterization using widgets of the catalog name without chnging the code or logic.
* **Aligned with IT Services Business Model:** 
The Schemas I have designed that is besically are the business domains that Cyntexa besically works on, for example sales, enginnering and finance besically are major domains, so it aligns with the IT service and business models.
* **Granular Role-Based Access Control (RBAC):**
Sensitive domains like `hr` and `finance` can be strictly Isolated off using schema-level access using the governance feature that allows us to give access based on domain and more, we can also give more fine-grained access.
* Operational domains like `engineering` and `sales` can be shared with project managers and business development teams without exposing proprietary financial data.

**9. Write a data-masking strategy: which columns need masking, which role tiers should see unmasked
data, and how Unity Catalog permissions would enforce it (this connects forward to Day 8
governance).**

The columns that contain sensetive information should be masked, and only the people specific to the domain should only be able to see those information and we can do that using governance.
The columns that should be masked for example:

HR: base_salary, bonus_amount, bank_account_number, pan_card, aadhar_card, ect.

Finance: client_bank_account, credit_card_info, etc.

Engineering: client_contact_info, project_details, etc.

So above I have given some example of the info that should be masked and only the people in that particular domain should have the access of the raw data, also it can be only the head of the each department holds the info of the raw data and even the people under them and other get to see the masked data, as we can give the fine-grained access, using the databricks governance feature.


Unity Catalog enforces governance across the 3-level namespace (`catalog.schema.table`) using four primary enforcement mechanisms:

---

**1. Hierarchical Privilege Inheritance (RBAC)**

* Privileges granted at a higher tier automatically flow down the hierarchy, ensuring broad security boundaries with zero permission leaks, means even on the table level if someone has the access do select and any write access they can't do it, if they don't have the access of the catalog and schema. 
* Write Access given to the engineering team at the dev level, but it does't enforce at the deployment on the production level. So we can do fine-grained access also.

---

**2. Fine-Grained Dynamic Column Masking**

Instead of duplicating tables or creating separate views, Unity Catalog evaluates SQL User-Defined Functions (UDFs) at runtime using is the current user belong to the group:

* When an unauthorized role queries a table with masked columns, the compute engine dynamically swaps plaintext values with redacted or hashed strings.
* When an authorized user runs the exact same query, Unity Catalog passes the clear-text values.

---

**3. Row-Level Security (Row Filters)**

Row filters restrict which records a user can view based on their organizational identity or attributes:

* **Multi-Region / Practice Isolation:** A Practice Lead for DataBricks only sees consulting rows where `practice_area = 'DataBricks'`.
* The row filter user defined function returns `TRUE` or `FALSE` for each row at query execution time, ensuring unauthorized rows are omitted entirely from query scans.

---

**4. Automated Lineage & Centralized Audit Logging**

* **Lineage Tracking:** Unity Catalog automatically captures column-level and table-level lineage across all SQL notebooks, workflows, and BI queries. If sensitive HR or client Personally Identifiable Information flows into a downstream reporting table, the governance lineage tracks its source.
* **System Audit Tables:** Every data modification, and export event is logged in `system.access.audit`, enabling compliance verification and audit discovery.

**10. (Data Analyst) Using samples.tpch, write a query with at least one CTE and one window function to
produce a 'top 5 customers by revenue per region' report.**

In [0]:
%sql
WITH customer_regional_revenue AS (
    SELECT r.r_name AS region_name, c.c_custkey AS customer_id, c.c_name AS customer_name, 
        SUM(o.o_totalprice) AS total_revenue
    FROM samples.tpch.customer c
    INNER JOIN samples.tpch.orders o ON c.c_custkey = o.o_custkey
    INNER JOIN samples.tpch.nation n ON c.c_nationkey = n.n_nationkey
    INNER JOIN samples.tpch.region r ON n.n_regionkey = r.r_regionkey
    GROUP BY r.r_name, c.c_custkey, c.c_name
),
ranked_customers AS (
    SELECT region_name, customer_id, customer_name,
        ROUND(total_revenue, 2) AS total_revenue,
        DENSE_RANK() OVER (
            PARTITION BY region_name 
            ORDER BY total_revenue DESC
        ) AS revenue_rank
    FROM customer_regional_revenue
)
SELECT region_name, revenue_rank, customer_id, customer_name, total_revenue
FROM ranked_customers
WHERE revenue_rank <= 5
ORDER BY region_name, revenue_rank;